# Group Isoforms into C-terminal and not C-terminal (Internal and N-terminal)
In the final database, we will add the entire new C-terminus for the isoforms that have a new C-terminus (can include several tryptic cleavage sites). For isoforms sharing a C-terminus with the canonical form, we will perform in-silico tryptic digestion and only add a unique fragment to the database (if we can find one).

# Setup
## Import and install Packages

In [ ]:
import sys
import os
import session_info

In [ ]:
import pandas as pd
import re
import difflib

In [ ]:
# Display session information
session_info.show()

## Set working directory

In [ ]:
cwd = os.getcwd()
if not cwd.endswith("Isoform_Database/Jupyter_environment/Isoform_Database_SIAF/02_Isoform_Database"):
    print("WARNING: The working directory is not set to the '02_Isoform_Database'!")
    print("Current working directory:", cwd)
else:
    print(f"Working directory is correctly set to the '02_Isoform_Database' folder (\"{cwd}\").")

In [ ]:
# Data directories
mapping_dir = "../01_UniProt/data/mapping/"
unique_dir = "../01_UniProt/data/unique/"

## Read in fasta files and mapping

In [ ]:
full_proteome_unique = pd.read_csv(os.path.join(unique_dir, 'full_proteome_unique.csv'))

iso_canonical_mapping = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping.csv'))
iso_canonical_mapping_flat = pd.read_csv(os.path.join(mapping_dir, 'iso_canonical_mapping_flat.csv'))

# Group Isoforms into C-terminal and not C-terminal

In [ ]:
# fast lookup dictionary from unique proteome dataframe
seq_lookup = full_proteome_unique.set_index('ID')['seq'].to_dict()

def process_isoforms_with_alignment_v2(mapping_df, seq_dict, pattern):
    results_list = []
    processed_isoforms = set()

    for _, row in mapping_df.iterrows():
        c_id = row['Entry']
        i_id = row['Isoform_ID']

        if i_id in processed_isoforms:
            continue
            
        c_seq, i_seq = seq_dict.get(c_id), seq_dict.get(i_id)
        if not c_seq or not i_seq: 
            continue

        # 1. Map out all tryptic cleavage boundaries in the ISOFORM sequence
        cuts = [0] + [m.end() for m in pattern.finditer(i_seq)]
        if cuts[-1] != len(i_seq):
            cuts.append(len(i_seq))

        # 2. CRITICAL: Force autojunk=False to ensure precise biological alignment
        matcher = difflib.SequenceMatcher(None, c_seq, i_seq, autojunk=False)
        
        for tag, c_start, c_end, i_start, i_end in matcher.get_opcodes():
            
            # Include 'delete' to capture clean, in-frame deletions
            if tag in ('insert', 'replace', 'delete'):
                
                # 3. Find flanking tryptic cut sites
                start_cut = max([c for c in cuts if c <= i_start])
                end_cut = min([c for c in cuts if c >= i_end])
                
                # Safety Edge Case: If a pure deletion lands precisely ON a tryptic cut site,
                # start_cut and end_cut will be equal. Expand to neighboring cuts to get the junction.
                if start_cut == end_cut:
                    idx = cuts.index(start_cut)
                    start_cut = cuts[idx - 1] if idx > 0 else cuts[0]
                    end_cut = cuts[idx + 1] if idx < len(cuts) - 1 else cuts[-1]
                
                # 4. Extract the true physical junction peptide
                variant_peptide = i_seq[start_cut:end_cut]
                is_true_c_term = (end_cut == len(i_seq))

                results_list.append({
                    'Isoform_ID': i_id,
                    'Canonical_ID': c_id,
                    'Mutation_Type': tag.capitalize(),
                    'Location_Type': 'C-Terminal Tail' if is_true_c_term else 'Internal Junction',
                    'Canonical_Mutation_Coords': f"{c_start + 1}-{c_end}",
                    'Isoform_Mutation_Coords': f"{i_start + 1}-{i_end}" if tag != 'delete' else "N/A (Deleted)",
                    'Captured_Peptide': variant_peptide,
                    'Peptide_Length': len(variant_peptide),
                    'Peptide_Coordinates_In_Isoform': f"{start_cut + 1}-{end_cut}"
                })
                
        processed_isoforms.add(i_id)
            
    return pd.DataFrame(results_list)

# Execution
tryP_pattern = re.compile(r'[KR](?!P)')
df_aligned_variants = process_isoforms_with_alignment_v2(iso_canonical_mapping_flat, seq_lookup, tryP_pattern)

In [ ]:
df_aligned_variants

In [ ]:
def collapse_isoform_peptides(aligned_df):
    """
    Collapses the aligned variants DataFrame so that each unique 
    combination of Isoform and Peptide appears exactly once.
    """
    if aligned_df.empty:
        return pd.DataFrame(columns=['Isoform_ID', 'Canonical_ID', 'Captured_Peptide', 'Peptide_Length', 'Occurrences', 'All_Coordinates'])

    # Group by Isoform and Peptide to collapse duplicates
    collapsed = aligned_df.groupby(['Isoform_ID', 'Canonical_ID', 'Captured_Peptide']).agg({
        'Peptide_Length': 'first',
        'Mutation_Type': lambda x: ', '.join(sorted(set(x))),
        'Peptide_Coordinates_In_Isoform': lambda x: ', '.join(x),
        # Use any existing column to get the total count of occurrences
        'Location_Type': 'count' 
    }).reset_index()

    # Rename columns for clarity
    collapsed = collapsed.rename(columns={
        'Location_Type': 'Occurrences',
        'Peptide_Coordinates_In_Isoform': 'All_Coordinates',
        'Mutation_Type': 'Aggregated_Mutation_Types'
    })

    # Reorder columns to put Isoform and Peptide front and center
    column_order = [
        'Isoform_ID', 
        'Canonical_ID', 
        'Captured_Peptide', 
        'Peptide_Length', 
        'Occurrences', 
        'All_Coordinates',
        'Aggregated_Mutation_Types'
    ]
    
    return collapsed[column_order]

# Execution
df_collapsed = collapse_isoform_peptides(df_aligned_variants)

In [ ]:
df_collapsed

In [ ]:
df_len_filtered = df_collapsed[df_collapsed["Peptide_Length"] >= 7]
df_len_filtered = df_len_filtered[df_len_filtered["Peptide_Length"] <= 52]

df_len_filtered